# Цель — привести labeled_pairs.parquet в нормальный вид перед baseline и ML.

In [1]:
import pandas as pd
import sys

sys.path.append('..')
from src.features.pair_dataset import (
    remove_duplicate_pairs,
    split_pairs_by_entity_group,
    print_pair_summary,
)

In [2]:
# ============================================================
# LOAD LABELED PAIRS
# ============================================================

pairs = pd.read_parquet(
    "../data/processed/labeled_pairs.parquet"
)

print(f"Pairs loaded: {len(pairs):,}")
display(pairs.head())

Pairs loaded: 151,014


,idx1,idx2,label,pair_type,name_similarity,record_id_1,party_public_id_1,party_name_1,name_latin_1,name_no_legal_1,...,party_public_id_2,party_name_2,name_latin_2,name_no_legal_2,country_2,script_2,relation_kind_2,relation_role_2,company_public_id_2,company_name_norm_2
0,3670911,3701401,0,easy_negative_different_public_id,NaN,tjk_rec_6b4e838005267448452f5c1133aa4c4d,tjk_party_b045fbfadecdf163,Катабеков Нурмурод Шодимуродович,katabekov nurmurod shodimurodovich,катабеков нурмурод шодимуродович,...,tjk_party_f016271677ad78d6,Нуманов Фахриддин Манонович,numanov fahriddin manonovich,нуманов фахриддин манонович,tjk,cyrillic,director,director,tjk_company_b5e0bd84c92681ff,муассисаи таҳсилоти миёнаи умумии No7
1,334475,244796,0,hard_negative_similar_name,90.322581,arm_rec_38b312f23f109b4ed6adbb069d2b866a,arm_party_ecf2ba27d1eca9fc,«ԱՄՓ ՀՈԼԴԻՆԳ» ՍՊԸ,amp holding spy,ամփ հոլդինգ սպը,...,arm_party_b9e7bd6a8eda69ac,«ԱՄՓ ՀՈԼԴԻՆԳՍ» ՓԲԸ,amp holdings pby,ամփ հոլդինգս փբը,arm,armenian,shareholder,beneficial_owner,arm_company_d2dba87b64a78c27,զանգեզուրի պղնձամոլիբդենային կոմբինատ
2,181156,445560,0,hard_negative_similar_name,90.322581,arm_rec_334e40b0e3226a3cb9c4e73b7cb2cd59,arm_party_ecf2ba27d1eca9fc,«ԱՄՓ ՀՈԼԴԻՆԳ» ՍՊԸ,amp holding spy,ամփ հոլդինգ սպը,...,arm_party_b9e7bd6a8eda69ac,«ԱՄՓ ՀՈԼԴԻՆԳՍ» ՓԲԸ,amp holdings pby,ամփ հոլդինգս փբը,arm,armenian,shareholder,beneficial_owner,arm_company_d2dba87b64a78c27,զանգեզուրի պղնձամոլիբդենային կոմբինատ
3,414456,244792,0,hard_negative_similar_name,90.322581,arm_rec_e2dc59d0be787068f6aabbcd86a22df0,arm_party_b9e7bd6a8eda69ac,«ԱՄՓ ՀՈԼԴԻՆԳՍ» ՓԲԸ,amp holdings pby,ամփ հոլդինգս փբը,...,arm_party_ecf2ba27d1eca9fc,«ԱՄՓ ՀՈԼԴԻՆԳ» ՍՊԸ,amp holding spy,ամփ հոլդինգ սպը,arm,armenian,shareholder,beneficial_owner,arm_company_081b4ce85680db74,զանգեզուրի պղնձամոլիբդենային կոմբինատ փբը
4,335053,171471,0,hard_negative_similar_name,90.322581,arm_rec_42a1de0414cde58f8791666c966cdb5f,arm_party_ecf2ba27d1eca9fc,«ԱՄՓ ՀՈԼԴԻՆԳ» ՍՊԸ,amp holding spy,ամփ հոլդինգ սպը,...,arm_party_b9e7bd6a8eda69ac,«ԱՄՓ ՀՈԼԴԻՆԳՍ» ՓԲԸ,amp holdings pby,ամփ հոլդինգս փբը,arm,armenian,shareholder,beneficial_owner,arm_company_d2dba87b64a78c27,զանգեզուրի պղնձամոլիբդենային կոմբինատ


In [3]:
# ============================================================
# INITIAL PAIR SUMMARY
# ============================================================

print_pair_summary(
    pairs,
    name="initial labeled pairs"
)

INITIAL LABELED PAIRS
Rows: 151,014

Label distribution:
label
0    100676
1     50338
Name: count, dtype: int64

Label ratio:
label
0    66.67
1    33.33
Name: proportion, dtype: float64

Pair types:
pair_type
easy_negative_different_public_id    50338
hard_negative_similar_name           50338
positive_same_public_id              50338
Name: count, dtype: int64


In [4]:
# ============================================================
# REMOVE DUPLICATE PAIRS
# ============================================================

pairs_clean = remove_duplicate_pairs(pairs)

Pairs before deduplication: 151,014
Pairs after deduplication:  151,012
Removed: 2


In [5]:
# ============================================================
# CLEANED PAIR SUMMARY
# ============================================================

print_pair_summary(
    pairs_clean,
    name="cleaned labeled pairs"
)

CLEANED LABELED PAIRS
Rows: 151,012

Label distribution:
label
0    100674
1     50338
Name: count, dtype: int64

Label ratio:
label
0    66.67
1    33.33
Name: proportion, dtype: float64

Pair types:
pair_type
easy_negative_different_public_id    50338
positive_same_public_id              50338
hard_negative_similar_name           50336
Name: count, dtype: int64


In [6]:
# ============================================================
# CHECK DUPLICATE PAIR KEYS
# ============================================================

duplicate_pair_keys = pairs_clean["pair_key"].duplicated().sum()

print(f"Duplicate pair keys: {duplicate_pair_keys}")

Duplicate pair keys: 0


In [7]:
# ============================================================
# SPLIT INTO TRAIN / VALID / TEST
# ============================================================

train_pairs, valid_pairs, test_pairs = split_pairs_by_entity_group(
    pairs_clean,
    test_size=0.15,
    valid_size=0.15,
    random_state=42,
)

print_pair_summary(train_pairs, name="train pairs")
print_pair_summary(valid_pairs, name="valid pairs")
print_pair_summary(test_pairs, name="test pairs")

TRAIN PAIRS
Rows: 120,894

Label distribution:
label
0    85619
1    35275
Name: count, dtype: int64

Label ratio:
label
0    70.82
1    29.18
Name: proportion, dtype: float64

Pair types:
pair_type
hard_negative_similar_name           50062
easy_negative_different_public_id    35557
positive_same_public_id              35275
Name: count, dtype: int64
VALID PAIRS
Rows: 15,306

Label distribution:
label
1    7758
0    7548
Name: count, dtype: int64

Label ratio:
label
1    50.69
0    49.31
Name: proportion, dtype: float64

Pair types:
pair_type
positive_same_public_id              7758
easy_negative_different_public_id    7409
hard_negative_similar_name            139
Name: count, dtype: int64
TEST PAIRS
Rows: 14,812

Label distribution:
label
0    7507
1    7305
Name: count, dtype: int64

Label ratio:
label
0    50.68
1    49.32
Name: proportion, dtype: float64

Pair types:
pair_type
easy_negative_different_public_id    7372
positive_same_public_id              7305
hard_negative_simil

In [8]:
# ============================================================
# LEAKAGE CHECK
# ============================================================

train_groups = set(train_pairs["entity_group"])
valid_groups = set(valid_pairs["entity_group"])
test_groups = set(test_pairs["entity_group"])

print("Train ∩ Valid:", len(train_groups & valid_groups))
print("Train ∩ Test:", len(train_groups & test_groups))
print("Valid ∩ Test:", len(valid_groups & test_groups))

Train ∩ Valid: 0
Train ∩ Test: 0
Valid ∩ Test: 0


In [9]:
# ============================================================
# SAVE SPLITS
# ============================================================

train_path = "../data/processed/train_pairs.parquet"
valid_path = "../data/processed/valid_pairs.parquet"
test_path = "../data/processed/test_pairs.parquet"

train_pairs.to_parquet(train_path, index=False)
valid_pairs.to_parquet(valid_path, index=False)
test_pairs.to_parquet(test_path, index=False)

print("Saved:")
print(train_path)
print(valid_path)
print(test_path)

Saved:
../data/processed/train_pairs.parquet
../data/processed/valid_pairs.parquet
../data/processed/test_pairs.parquet


## Вывод по разделению пар

После очистки датасета пар осталось 151 012 пар. Дубликаты практически отсутствовали: было удалено только 2 повторяющиеся пары.

Разделение на train/valid/test выполнено по группам сущностей, а не случайно по строкам. Это важно, потому что снижает риск утечки информации между обучением и проверкой качества. Проверка показала, что пересечений между train, valid и test нет.

При этом обнаружено ограничение текущего разделения: большая часть сложных отрицательных примеров попала в train, а в valid и test осталось мало hard negative pairs. Поэтому первые метрики baseline могут быть несколько завышены. В следующих итерациях потребуется сделать стратифицированное разделение по типам пар.